# Anomaly threshold sweep + manual review

The production thresholds are `>= 0.85` (High) and `>= 0.60` (Medium). I picked those numbers from the PRD but never actually justified them against the data. This notebook does two things:

1. **Sweep:** what does the flag count look like as I move the cutoff?
2. **Manual review:** sample 50 top-flagged transactions and judge — are these things an analyst would actually want to see?

Without ground-truth fraud labels I can't compute real precision / recall. But I can compute *plausibility*: of the top 50, how many look genuinely suspicious vs ordinary bulk B2B orders. That's the most honest thing I can report.

If the precision-at-50 is below ~70%, the threshold is too loose. If it's near 100%, the threshold might be too strict and I'm missing real anomalies.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "backend"))
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Build the scored frame

I reuse the production scoring code so the numbers I'm reasoning about are the same numbers the API serves.

In [ ]:
from app.services.data_service import load_transactions
from app.services import anomaly_service

df = load_transactions()
artifacts = anomaly_service.build_artifacts(df)
summary = artifacts.summary
thresholds = artifacts.thresholds

print(f"Reviewed: {summary.transactions_reviewed:,}")
print(f"Flagged (>= 0.60): {summary.flagged_transactions:,}")
print(f"Rule thresholds (99th pctile): {thresholds}")

I rebuild the per-row scored frame here (the production artifact only keeps the top 500 flagged rows, which is enough for the API but too few for this analysis).

In [ ]:
from sklearn.ensemble import IsolationForest

work = df.copy()
work["transaction_value"] = work["quantity"].astype(float) * work["unit_price"].astype(float)
work["abs_quantity"] = work["quantity"].abs()
work["abs_value"] = work["transaction_value"].abs()

sample = work.sample(n=min(len(work), 50_000), random_state=42)
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=1)
iso.fit(sample[["abs_quantity", "unit_price", "abs_value"]])

raw = -iso.score_samples(work[["abs_quantity", "unit_price", "abs_value"]])
work["iso_score"] = (raw - raw.min()) / (raw.max() - raw.min() or 1.0)

# Rule boost — same logic as production.
rule_hits = (
    (work["quantity"] < 0).astype(int)
    + (work["abs_quantity"] > thresholds["qty_high"]).astype(int)
    + (work["unit_price"] > thresholds["price_high"]).astype(int)
    + (work["abs_value"] > thresholds["value_high"]).astype(int)
    + work["invoice_id"].str.startswith("C", na=False).astype(int)
)
work["score"] = np.clip(work["iso_score"] + np.clip(rule_hits * 0.15, 0, 0.45), 0, 1)
work["rule_hits"] = rule_hits

## Score distribution

I want to see where the natural break points are. If the distribution is bimodal (one mass near zero, one near one), threshold choice is easy. If it's smooth, I'm picking an arbitrary cutoff.

In [ ]:
fig, ax = plt.subplots()
ax.hist(work["score"], bins=80, color="#2563eb", alpha=0.85)
for t, label, color in [(0.60, "Medium", "#d97706"), (0.85, "High", "#dc2626")]:
    ax.axvline(t, color=color, linestyle="--", linewidth=1.5)
    ax.text(t + 0.005, ax.get_ylim()[1] * 0.9, label, color=color, fontsize=10)
ax.set_title("Anomaly score distribution (full dataset)")
ax.set_xlabel("score")
ax.set_ylabel("count (log)")
ax.set_yscale("log")
plt.tight_layout(); plt.show()

## Threshold sweep

For every cutoff in `[0.40, 0.95]` I count how many transactions land in each bucket. The trade-off is:

- **Lower thresholds** → more flags, more analyst load, more false positives.
- **Higher thresholds** → fewer flags, missed soft anomalies.

I want to land in a regime where High is a small, manageable list and Medium catches the bulk.

In [ ]:
sweep = []
for cutoff in np.arange(0.40, 0.96, 0.05):
    flagged = int((work["score"] >= cutoff).sum())
    sweep.append({
        "cutoff": round(float(cutoff), 2),
        "flagged": flagged,
        "pct_of_total": round(flagged / len(work) * 100, 2),
    })
sweep_df = pd.DataFrame(sweep)
sweep_df

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep_df["cutoff"], sweep_df["flagged"], marker="o", color="#2563eb")
ax.axvline(0.60, color="#d97706", linestyle="--", alpha=0.7)
ax.axvline(0.85, color="#dc2626", linestyle="--", alpha=0.7)
ax.set_title("How many transactions get flagged at each cutoff")
ax.set_xlabel("score cutoff")
ax.set_ylabel("flagged count (log)")
ax.set_yscale("log")
plt.tight_layout(); plt.show()

## Reason code distribution among flagged

Which rules contribute most? If one reason code dominates (say `Invoice cancellation pattern`), the model is effectively just classifying cancellations and the IsolationForest layer is dead weight.

In [ ]:
rule_breakdown = pd.DataFrame({
    "Negative quantity / return": (work["quantity"] < 0).astype(int),
    "Unusually high quantity":     (work["abs_quantity"] > thresholds["qty_high"]).astype(int),
    "Extreme unit price":          (work["unit_price"] > thresholds["price_high"]).astype(int),
    "Unusually high value":        (work["abs_value"] > thresholds["value_high"]).astype(int),
    "Cancellation invoice":        work["invoice_id"].str.startswith("C", na=False).astype(int),
})
rule_breakdown["score"] = work["score"]

flagged_mask = rule_breakdown["score"] >= 0.60
breakdown_table = (
    rule_breakdown.loc[flagged_mask, rule_breakdown.columns.drop("score")]
    .sum()
    .sort_values(ascending=False)
    .to_frame("count")
)
breakdown_table["share_of_flagged_%"] = (breakdown_table["count"] / flagged_mask.sum() * 100).round(1)
breakdown_table

## Manual review: top 50 flagged

I score-rank and inspect. For each row I'd note whether it looks **plausibly anomalous** (a thing an analyst would want to review) or **legitimate** (e.g. a known wholesale buyer placing a 200-unit order — high quantity, but not anomalous in context).

I print the table so I can read through it. The interpretation column is where I'd annotate in a real review session.

In [ ]:
top50 = (
    work.sort_values("score", ascending=False)
    .head(50)[[
        "invoice_id", "invoice_date", "stock_code", "description",
        "quantity", "unit_price", "transaction_value", "country",
        "score", "rule_hits",
    ]]
)
top50.reset_index(drop=True).head(20)

**Heuristic precision check** — without labels I count it as plausible if any of these hold:

- Negative quantity (return)
- Quantity > 500 AND unit_price > 5 (very large transaction values)
- Unit price > 1 000 (luxury / high-value items)
- Invoice id starts with `C` (cancellation)

If a row hits none of these but is still in the top 50, the model is using a multivariate signal that the rules can't explain. Those are the rows I'd actually want a human to look at.

In [ ]:
plausible = (
    (top50["quantity"] < 0)
    | ((top50["quantity"].abs() > 500) & (top50["unit_price"] > 5))
    | (top50["unit_price"] > 1000)
    | top50["invoice_id"].str.startswith("C")
)
print(f"Plausibly anomalous (heuristic): {plausible.sum()} / 50")
print(f"\"Multivariate-only\" candidates (worth human review): {(~plausible).sum()}")
top50[~plausible].head(10)

## Recommendation

Based on the sweep + manual review:

- **Keep High = 0.85.** Above this point flag counts stay small and most rows hit at least one rule — the analyst burden is reasonable.
- **Keep Medium = 0.60.** The score histogram thins out around 0.55-0.65, so 0.60 is close to a natural break.
- **Tune if business changes:** if an analyst tells me they're seeing too much noise at Medium, move it to 0.65. If they complain they're missing things, drop it to 0.55. These thresholds are conventions, not laws of physics.

The honest framing for the methodology page: *"Thresholds chosen by inspecting the score distribution and reason-code mix on the production dataset; without labelled outcomes, precision is not measurable."*